In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Cluster Heartbeat - Data Exploration\n",
    "\n",
    "## Exploring GPU Cluster Metrics Data\n",
    "\n",
    "This notebook explores the synthetic and real cluster metrics data for the Cluster Heartbeat system."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "from pathlib import Path\n",
    "\n",
    "# Add parent directory to path\n",
    "sys.path.insert(0, str(Path.cwd().parent))\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from datetime import datetime\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set plotting style\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette(\"husl\")\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import Cluster Heartbeat modules\n",
    "from src.config import load_config\n",
    "from src.data.ingestion import DataIngestion\n",
    "from src.data.synthetic_generator import SyntheticDataGenerator\n",
    "from src.data.preprocessing import DataPreprocessor"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load configuration\n",
    "config = load_config('../config/config.yaml')\n",
    "print(\"Configuration loaded successfully!\")\n",
    "print(f\"Environment: {config['project']['environment']}\")\n",
    "print(f\"Metrics: {len(config['features']['metrics'])} metrics\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate synthetic data\n",
    "generator = SyntheticDataGenerator(config)\n",
    "cluster_metrics = generator.generate()\n",
    "df = cluster_metrics.to_dataframe()\n",
    "\n",
    "print(f\"Generated {len(df)} data points\")\n",
    "print(f\"Columns: {', '.join(df.columns)}\")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Data statistics\n",
    "print(\"=\"*60)\n",
    "print(\"DATA STATISTICS\")\n",
    "print(\"=\"*60)\n",
    "print(f\"\\nTotal samples: {len(df)}\")\n",
    "print(f\"Total features: {len(df.columns)}\")\n",
    "print(f\"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB\")\n",
    "\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"DESCRIPTIVE STATISTICS\")\n",
    "print(\"=\"*60)\n",
    "df.describe()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check for null values\n",
    "null_counts = df.isnull().sum()\n",
    "if null_counts.any():\n",
    "    print(\"Null values detected:\")\n",
    "    print(null_counts[null_counts > 0])\n",
    "else:\n",
    "    print(\"No null values found!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Time series visualization\n",
    "fig, axes = plt.subplots(3, 2, figsize=(15, 12))\n",
    "axes = axes.flatten()\n",
    "\n",
    "metrics_to_plot = ['gpu_utilization', 'memory_utilization', 'gpu_temperature', \n",
    "                   'power_consumption', 'cpu_usage', 'network_throughput']\n",
    "\n",
    "for i, metric in enumerate(metrics_to_plot):\n",
    "    if metric in df.columns and i < len(axes):\n",
    "        axes[i].plot(df[metric].values[:500], label=metric, alpha=0.7)\n",
    "        axes[i].set_title(metric, fontsize=12)\n",
    "        axes[i].set_xlabel('Time Steps')\n",
    "        axes[i].legend()\n",
    "        axes[i].grid(True, alpha=0.3)\n",
    "\n",
    "# Hide empty subplots\n",
    "for j in range(i+1, len(axes)):\n",
    "    axes[j].set_visible(False)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Correlation matrix\n",
    "numeric_cols = [col for col in df.columns if col not in ['timestamp', 'node_id', 'job_id']]\n",
    "corr_matrix = df[numeric_cols].corr()\n",
    "\n",
    "plt.figure(figsize=(12, 10))\n",
    "mask = np.triu(np.ones_like(corr_matrix, dtype=bool))\n",
    "sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', \n",
    "            center=0, square=True, linewidths=0.5)\n",
    "plt.title('Correlation Matrix of Cluster Metrics', fontsize=14)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Distribution of metrics\n",
    "fig, axes = plt.subplots(3, 3, figsize=(15, 12))\n",
    "axes = axes.flatten()\n",
    "\n",
    "for i, metric in enumerate(numeric_cols[:9]):\n",
    "    if i < len(axes):\n",
    "        axes[i].hist(df[metric].values, bins=50, alpha=0.7, edgecolor='black')\n",
    "        axes[i].set_title(metric, fontsize=12)\n",
    "        axes[i].set_xlabel('Value')\n",
    "        axes[i].set_ylabel('Frequency')\n",
    "\n",
    "for j in range(i+1, len(axes)):\n",
    "    axes[j].set_visible(False)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Node distribution\n",
    "if 'node_id' in df.columns:\n",
    "    node_counts = df['node_id'].value_counts()\n",
    "    \n",
    "    plt.figure(figsize=(10, 6))\n",
    "    node_counts.plot(kind='bar', color='skyblue', edgecolor='black')\n",
    "    plt.title('Data Distribution by Node', fontsize=14)\n",
    "    plt.xlabel('Node ID')\n",
    "    plt.ylabel('Count')\n",
    "    plt.xticks(rotation=45)\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    \n",
    "    print(f\"Total nodes: {len(node_counts)}\")\n",
    "    print(f\"Nodes: {sorted(node_counts.index.tolist())}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Identify anomalies in data\n",
    "from sklearn.ensemble import IsolationForest\n",
    "\n",
    "# Prepare data for anomaly detection\n",
    "features = df[numeric_cols].values\n",
    "\n",
    "# Fit Isolation Forest\n",
    "iso_forest = IsolationForest(contamination=0.05, random_state=42)\n",
    "anomaly_labels = iso_forest.fit_predict(features)\n",
    "anomaly_scores = iso_forest.score_samples(features)\n",
    "\n",
    "# Plot anomalies\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# Anomaly scores\n",
    "axes[0].hist(anomaly_scores, bins=50, alpha=0.7, edgecolor='black')\n",
    "axes[0].axvline(x=np.percentile(anomaly_scores, 5), color='red', linestyle='--', \n",
    "                label='Anomaly threshold (5%)')\n",
    "axes[0].set_title('Anomaly Score Distribution', fontsize=12)\n",
    "axes[0].set_xlabel('Anomaly Score')\n",
    "axes[0].set_ylabel('Frequency')\n",
    "axes[0].legend()\n",
    "\n",
    "# Anomaly count\n",
    "anomaly_count = np.sum(anomaly_labels == -1)\n",
    "axes[1].bar(['Normal', 'Anomaly'], [len(anomaly_labels) - anomaly_count, anomaly_count], \n",
    "            color=['green', 'red'], edgecolor='black')\n",
    "axes[1].set_title(f'Anomalies Detected: {anomaly_count} ({anomaly_count/len(anomaly_labels)*100:.1f}%)', \n",
    "                  fontsize=12)\n",
    "axes[1].set_ylabel('Count')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Summary\n",
    "print(\"=\"*60)\n",
    "print(\"DATA EXPLORATION SUMMARY\")\n",
    "print(\"=\"*60)\n",
    "print(f\"\\n✅ Total samples: {len(df)}\")\n",
    "print(f\"✅ Total features: {len(df.columns)}\")\n",
    "print(f\"✅ Null values: {df.isnull().sum().sum()}\")\n",
    "print(f\"✅ Unique nodes: {len(df['node_id'].unique()) if 'node_id' in df.columns else 'N/A'}\")\n",
    "print(f\"✅ Unique jobs: {len(df['job_id'].unique()) if 'job_id' in df.columns else 'N/A'}\")\n",
    "print(f\"✅ Anomalies detected: {anomaly_count} ({anomaly_count/len(anomaly_labels)*100:.1f}%)\")\n",
    "\n",
    "print(\"\\n📊 Data ready for feature engineering!\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}